# 02 — Customer Intelligence: 10 Business SQL Queries

**Dataset:** 67,325 real Amazon Electronics reviews (5-core subset)
**Engine:** SQLite in-memory
**Source:** Stanford SNAP — http://snap.stanford.edu/data/amazon/

In [1]:
import pandas as pd
import sqlite3

# Load real data
df = pd.read_csv('../data/amazon_reviews_electronics_5core.csv')
df['reviewDate'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['reviewYear'] = df['reviewDate'].dt.year
df['reviewMonth'] = df['reviewDate'].dt.month
df['reviewLength'] = df['reviewText'].fillna('').str.len()

conn = sqlite3.connect(':memory:')
df.to_sql('reviews', conn, index=False, if_exists='replace')

print(f"Loaded {len(df):,} rows")
print(f"Date range: {df.reviewDate.min().date()} to {df.reviewDate.max().date()}")
print(f"Products: {df.asin.nunique():,} | Reviewers: {df.reviewerID.nunique():,}")

Loaded 67,325 rows
Date range: 1999-11-23 to 2014-07-23
Products: 27,832 | Reviewers: 53,609


## Q1 — Product Performance Ranking (min 10 reviews)

In [2]:
q = '''SELECT
    asin AS product_id,
    COUNT(*) AS review_volume,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    SUM(helpful_total) AS total_votes
FROM reviews
GROUP BY asin
HAVING COUNT(*) >= 10
ORDER BY avg_rating DESC, review_volume DESC
LIMIT 20'''
pd.read_sql(q, conn)

,product_id,review_volume,avg_rating,avg_helpfulness,total_votes
0,B0040JHVC2,18,5.00,0.72,34
1,B0009YDP7W,13,5.00,1.00,22
2,B0060B7NCG,13,5.00,1.00,2
3,B001TH7GUK,11,5.00,1.00,5
4,B003SX0P1A,11,5.00,1.00,2
5,B00007GQLU,10,5.00,0.89,22
6,B002IO2UM2,10,5.00,0.56,10
7,B003RQBKLC,10,5.00,0.86,90
8,B00081A2KY,19,4.95,0.67,3
9,B006U3O566,19,4.95,0.88,25


## Q2 — Rating Distribution by Year

In [3]:
q = '''SELECT
    reviewYear AS year,
    ROUND(AVG(overall), 2) AS avg_rating,
    COUNT(*) AS review_count,
    ROUND(AVG(reviewLength), 0) AS avg_length_chars
FROM reviews
WHERE reviewYear >= 2005
GROUP BY reviewYear
ORDER BY reviewYear'''
pd.read_sql(q, conn)

,year,avg_rating,review_count,avg_length_chars
0,2005,3.91,376,1190.0
1,2006,3.91,580,1092.0
2,2007,4.14,1343,784.0
3,2008,4.15,2006,889.0
4,2009,4.11,2789,947.0
5,2010,4.12,4064,928.0
6,2011,4.12,6950,809.0
7,2012,4.21,11320,689.0
8,2013,4.28,23704,514.0
9,2014,4.28,13625,471.0


## Q3 — Reviewer Loyalty: Repeat vs One-Time

In [4]:
q = '''SELECT
    CASE WHEN review_count = 1 THEN 'One-time'
         WHEN review_count <= 3 THEN 'Casual (2-3)'
         ELSE 'Loyal (4+)' END AS reviewer_type,
    COUNT(*) AS reviewer_count,
    ROUND(AVG(avg_rating), 2) AS avg_rating_given,
    SUM(review_count) AS total_reviews
FROM (
    SELECT reviewerID, COUNT(*) AS review_count, AVG(overall) AS avg_rating
    FROM reviews
    GROUP BY reviewerID
)
GROUP BY reviewer_type
ORDER BY total_reviews DESC'''
pd.read_sql(q, conn)

,reviewer_type,reviewer_count,avg_rating_given,total_reviews
0,One-time,43568,4.21,43568
1,Casual (2-3),9337,4.23,20131
2,Loyal (4+),704,4.27,3626


## Q4 — Seasonal Patterns (Monthly Aggregates)

In [5]:
q = '''SELECT
    reviewMonth AS month,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(reviewLength), 0) AS avg_length
FROM reviews
GROUP BY reviewMonth
ORDER BY reviewMonth'''
pd.read_sql(q, conn)

,month,review_count,avg_rating,avg_length
0,1,7661,4.24,593.0
1,2,6158,4.27,596.0
2,3,6272,4.25,612.0
3,4,5882,4.21,620.0
4,5,5681,4.24,640.0
5,6,5639,4.21,657.0
6,7,5291,4.19,624.0
7,8,4248,4.18,677.0
8,9,3994,4.19,720.0
9,10,4197,4.20,704.0


## Q5 — Verified Purchase Proxy: High vs Low Helpfulness

In [6]:
q = '''SELECT
    CASE WHEN helpful_total >= 5 THEN 'High engagement (5+ votes)'
         WHEN helpful_total > 0 THEN 'Some engagement (1-4)'
         ELSE 'No engagement' END AS engagement_level,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(reviewLength), 0) AS avg_length
FROM reviews
GROUP BY engagement_level
ORDER BY review_count DESC'''
pd.read_sql(q, conn)

,engagement_level,review_count,avg_rating,avg_length
0,No engagement,38467,4.40,409.0
1,Some engagement (1-4),20647,4.08,727.0
2,High engagement (5+ votes),8211,3.72,1490.0


## Q6 — Product Lifecycle: Early vs Mature Reviews

In [7]:
q = '''WITH ranked AS (
    SELECT
        asin, overall, reviewDate,
        helpful_total, helpful_upvotes,
        ROW_NUMBER() OVER (PARTITION BY asin ORDER BY reviewDate) AS seq,
        COUNT(*) OVER (PARTITION BY asin) AS total
    FROM reviews
)
SELECT
    CASE WHEN seq <= 3 THEN 'Early (1st-3rd)'
         WHEN seq <= 10 THEN 'Growth (4th-10th)'
         ELSE 'Mature (11th+)' END AS stage,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness
FROM ranked
WHERE total >= 5
GROUP BY stage
ORDER BY MIN(seq)'''
pd.read_sql(q, conn)

,stage,review_count,avg_rating,avg_helpfulness
0,Early (1st-3rd),8115,4.27,0.75
1,Growth (4th-10th),11983,4.27,0.72
2,Mature (11th+),8243,4.41,0.69


## Q7 — Review Length Impact on Helpfulness

In [8]:
q = '''SELECT
    CASE WHEN reviewLength < 200 THEN 'Short (<200 chars)'
         WHEN reviewLength < 500 THEN 'Medium (200-500)'
         WHEN reviewLength < 1000 THEN 'Long (500-1000)'
         ELSE 'Very long (1000+)' END AS length_bucket,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(helpful_total), 1) AS avg_votes
FROM reviews
WHERE helpful_total > 0
GROUP BY length_bucket
ORDER BY MIN(reviewLength)'''
pd.read_sql(q, conn)

,length_bucket,review_count,avg_rating,avg_helpfulness,avg_votes
0,Short (<200 chars),4699,4.12,0.66,3.2
1,Medium (200-500),8584,3.98,0.72,3.9
2,Long (500-1000),7185,3.88,0.76,6.6
3,Very long (1000+),8390,3.97,0.82,18.4


## Q8 — Top Reviewers by Volume and Impact

In [9]:
q = '''SELECT
    reviewerID,
    reviewerName,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    SUM(helpful_upvotes) AS total_helpful_upvotes,
    SUM(helpful_total) AS total_votes_received
FROM reviews
GROUP BY reviewerID, reviewerName
HAVING COUNT(*) >= 5
ORDER BY review_count DESC
LIMIT 15'''
pd.read_sql(q, conn)

,reviewerID,reviewerName,review_count,avg_rating,total_helpful_upvotes,total_votes_received
0,ADLVFFE4VBT8,"A. Dent ""Aragorn""",23,4.52,738,775
1,A6FIAB28IS79,Samuel Chell,18,4.17,343,388
2,A3LGT6UZL99IW1,"Richard C. Drew ""Anaal Nathra/Uthe vas Bethod...",17,4.06,93,109
3,A5JLAU2ARJ0BO,"Gadgester ""No Time, No Money""",16,3.63,959,1080
4,A3AYSYSLHU26U9,Amazon Deity,15,4.47,61,65
5,A17BUUBOU0598B,"Mark ""Technology, Music and Movies""",14,3.36,692,738
6,A18HE80910BTZI,"rbhatta ""A Dinosaur you can trust!""",13,3.92,327,357
7,A22CW0ZHY3NJH8,Noname,13,4.15,301,321
8,A680RUE1FDO8B,Jerry Saperstein,13,4.69,59,63
9,ARBKYIVNYWK3C,RST10,13,4.69,57,66


## Q9 — Sentiment Proxy: 5-Star vs 1-Star Word Length

In [10]:
q = '''SELECT
    overall AS stars,
    COUNT(*) AS review_count,
    ROUND(AVG(reviewLength), 0) AS avg_length,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness
FROM reviews
WHERE overall IN (1, 5)
GROUP BY overall
ORDER BY overall DESC'''
pd.read_sql(q, conn)

,stars,review_count,avg_length,avg_helpfulness
0,5.0,40085,553.0,0.81
1,1.0,4377,642.0,0.58


## Q10 — Review Velocity: Products with Accelerating Reviews

In [11]:
q = '''WITH monthly AS (
    SELECT
        asin, reviewYear, reviewMonth,
        COUNT(*) AS monthly_reviews
    FROM reviews
    GROUP BY asin, reviewYear, reviewMonth
),
velocity AS (
    SELECT
        asin,
        reviewYear * 12 + reviewMonth AS period,
        monthly_reviews,
        LAG(monthly_reviews) OVER (PARTITION BY asin ORDER BY reviewYear, reviewMonth) AS prev_month
    FROM monthly
)
SELECT
    asin AS product_id,
    COUNT(*) AS months_reviewed,
    SUM(monthly_reviews) AS total_reviews,
    ROUND(AVG(monthly_reviews), 1) AS avg_monthly,
    SUM(CASE WHEN monthly_reviews > COALESCE(prev_month, 0) THEN 1 ELSE 0 END) AS accelerating_months
FROM velocity
GROUP BY asin
HAVING total_reviews >= 10
ORDER BY accelerating_months DESC, total_reviews DESC
LIMIT 15'''
pd.read_sql(q, conn)

,product_id,months_reviewed,total_reviews,avg_monthly,accelerating_months
0,B0019EHU8G,48,134,2.8,18
1,B0002L5R78,49,101,2.1,18
2,B003ES5ZUU,37,161,4.4,17
3,B001XURP7W,27,69,2.6,14
4,B000QUUFRW,27,87,3.2,13
5,B0041Q38NU,32,74,2.3,13
6,B007WTAJTO,24,187,7.8,12
7,B005DKZTMG,26,77,3.0,12
8,B000S5Q9CA,27,57,2.1,12
9,B005CT56F8,23,48,2.1,12


---
**Summary:** All 10 queries executed against 67,325 real Amazon Electronics reviews spanning 1999–2014.